# Financed Emissions — PCAF Accounting Layer (Model B)

Model A (`emissions.py`) answers a modeling question: *what does this vehicle
emit, in gCO2/km?* Model B answers an accounting question: *of that, how many
tonnes does the lender book against its loan portfolio, and how good is the
estimate?*

The two are decoupled on purpose. Model B consumes a frame that already carries a
`co2_estimate` column and never re-derives per-km emissions — so the regime logic
(EV = 0, PHEV corrected, combustion modelled) is decided once, in Model A.

**PCAF motor-vehicle-loans mechanics:**

| Quantity | Formula |
|---|---|
| Attribution factor | outstanding amount / vehicle value at origination |
| Vehicle emissions | annual distance (km) × emission factor (gCO2/km) |
| Financed emissions | attribution factor × vehicle emissions |

Reusable logic lives in `src/portfolio.py`; this notebook orchestrates it. See
`WRITEUP.md` for how the data-quality score maps onto the BNPPF / PCAF disclosure
problem.

## 1. Setup

In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import portfolio as pf      # Model B

## 2. Get per-vehicle emissions from Model A

The intended path: load the distinct specs, load the model trained in
`02_clean.ipynb`, and run the three-regime router to attach `co2_estimate`.

If you are running this notebook without the EEA data or the saved model (e.g. a
reviewer), set `DEMO = True` to synthesise a small spec table with the same
columns so the accounting layer still runs end-to-end.

In [2]:
DEMO = False   # True -> synthesise specs instead of loading real data + model

if not DEMO:
    import emissions as em      # Model A (needs EEA data + saved model)
    car_data = em.load_car_data(n=160_000, include_zr=True)
    ice_model = em.load_model("../models/ice_model.pkl")
    phev_median = em.phev_median_ewltp(car_data)
    car_data = car_data.copy()
    car_data["co2_estimate"] = em.estimate_co2_frame(car_data, ice_model, phev_median)
else:
    rng = np.random.default_rng(42)
    N = 8_000
    fuels = rng.choice(["petrol","diesel","electric","petrol/electric","hydrogen","lpg"],
                       size=N, p=[0.45,0.30,0.12,0.08,0.01,0.04])
    ec = rng.integers(900,3000,N); ep = rng.integers(50,220,N); m = rng.integers(1000,2200,N)
    co2 = pd.Series((60 + 0.03*ec + 0.15*ep + 0.02*m + rng.normal(0,6,N)).clip(70,260))
    ft = pd.Series(fuels)
    co2[ft.isin(["electric","hydrogen"])] = 0.0
    co2[ft.str.contains("/electric")] = 90.0
    car_data = pd.DataFrame({"Ft":fuels,"Ec (cm3)":ec,"Ep (KW)":ep,"M (kg)":m,
                             "Ewltp (g/km)":co2.values,"co2_estimate":co2.values})

print(car_data.shape)
car_data[["Ft","Ec (cm3)","Ep (KW)","M (kg)","co2_estimate"]].head()

(153739, 8)


,Ft,Ec (cm3),Ep (KW),M (kg),co2_estimate
0,diesel,NaN,12.0,NaN,151.989441
1,diesel,1968.0,90.0,NaN,147.976089
2,diesel,1968.0,142.0,NaN,156.869934
3,diesel,1995.0,110.0,NaN,133.089661
4,diesel,1997.0,81.0,NaN,177.053909


## 3. Attach a loan book

Model B needs a loan per vehicle: an `outstanding_amount` and an
`origination_value`. Real loan-level data lives in securitisation repositories
(US Reg-AB auto-ABS on SEC EDGAR; European DataWarehouse auto-ABS templates) —
plug those in here and the rest of the pipeline is unchanged. For a reproducible
demo we synthesise a plausible book (this is a stand-in, **not** a model).

In [3]:
book = pf.make_synthetic_portfolio(car_data, n_loans=6_000, seed=1)
book["attribution_factor"] = pf.attribution_factor(
    book["outstanding_amount"], book["origination_value"])
print("attribution factor summary:")
print(book["attribution_factor"].describe()[["mean","min","max"]])
book[["Ft","co2_estimate","origination_value","outstanding_amount","attribution_factor"]].head()

attribution factor summary:
mean    0.448066
min     0.039922
max     0.994608
Name: attribution_factor, dtype: float64


,Ft,co2_estimate,origination_value,outstanding_amount,attribution_factor
0,petrol/electric,28.750000,42000.0,23620.0,0.562381
1,electric,0.000000,68800.0,55050.0,0.800145
2,diesel,181.022842,44300.0,4420.0,0.099774
3,electric,0.000000,31500.0,17300.0,0.549206
4,diesel,205.412079,50400.0,16730.0,0.331944


## 4. Two PCAF methods, side by side

The whole point of Model A is a **data-quality upgrade**. To show it, we cost the
*same* portfolio two ways:

- **Baseline (PCAF score 5):** loan amount only. No vehicle known, so emissions
  come from an economic factor (tCO2e per euro). This is the honest fallback when
  you have nothing else — and note you cannot even report a gCO2/km intensity.
- **With Model A (PCAF score 3):** modelled gCO2/km per vehicle × assumed annual
  distance. Grounded in vehicle physics, and now intensity is reportable.

In [4]:
# --- Baseline: economic method, everything scored 5 ---
base = book.copy()
base["financed_tco2"] = pf.financed_emissions_economic(base)
base["pcaf_score"]    = pf.pcaf_score(["economic"] * len(base))
s_base = pf.portfolio_summary(base)

# --- With Model A: physical method, modelled specs scored 3 ---
mod = book.copy()
mod["financed_tco2"] = pf.financed_emissions_physical(mod)
mod["pcaf_score"]    = pf.pcaf_score(["modelled_specific"] * len(mod))
s_mod = pf.portfolio_summary(mod)

def show(tag, s, physical=True):
    print(f"{tag}")
    print(f"  loans               : {s['n_loans']:,}")
    print(f"  outstanding         : EUR {s['total_outstanding_eur']:,.0f}")
    print(f"  absolute financed   : {s['absolute_tco2e']:,.0f} tCO2e / yr")
    print(f"  economic intensity  : {s['econ_intensity_tco2e_per_eurM']:.1f} tCO2e / EUR-M")
    pi = f"{s['phys_intensity_gco2_km']:.0f} gCO2/km" if physical else "not reportable"
    print(f"  physical intensity  : {pi}")
    print(f"  wavg PCAF score     : {s['wavg_pcaf_score']:.2f}\n")

show("BASELINE  (loan amount only -> PCAF 5)", s_base, physical=False)
show("WITH MODEL A  (modelled specs -> PCAF 3)", s_mod, physical=True)

BASELINE  (loan amount only -> PCAF 5)
  loans               : 6,000
  outstanding         : EUR 120,438,120
  absolute financed   : 8,029 tCO2e / yr
  economic intensity  : 66.7 tCO2e / EUR-M
  physical intensity  : not reportable
  wavg PCAF score     : 5.00

WITH MODEL A  (modelled specs -> PCAF 3)
  loans               : 6,000
  outstanding         : EUR 120,438,120
  absolute financed   : 3,579 tCO2e / yr
  economic intensity  : 29.7 tCO2e / EUR-M
  physical intensity  : 117 gCO2/km
  wavg PCAF score     : 3.00



## 5. The upgrade, in one line

Same loans, same exposure — a better estimate and a better score. The economic
fallback and the physics-grounded estimate disagree materially, which is exactly
why data quality is not a cosmetic metric.

In [5]:
print(f"PCAF data quality : {s_base['wavg_pcaf_score']:.1f}  ->  {s_mod['wavg_pcaf_score']:.1f}")
print(f"absolute estimate : {s_base['absolute_tco2e']:,.0f}  ->  {s_mod['absolute_tco2e']:,.0f} tCO2e/yr "
      f"({100*(s_mod['absolute_tco2e']/s_base['absolute_tco2e']-1):+.0f}% vs the blunt method)")
print(f"reported intensity: {s_mod['phys_intensity_gco2_km']:.0f} gCO2/km")
print("peer references   : BNP Paribas group 141 | Arval 105 | Santander 133 gCO2/km")

PCAF data quality : 5.0  ->  3.0
absolute estimate : 8,029  ->  3,579 tCO2e/yr (-55% vs the blunt method)
reported intensity: 117 gCO2/km
peer references   : BNP Paribas group 141 | Arval 105 | Santander 133 gCO2/km


## 6. Regime spot-check\n\nConfirm each fuel regime flows through the accounting untouched.

In [6]:
for f in ["petrol","diesel","electric","petrol/electric"]:
    sub = mod[mod["Ft"] == f].head(1)
    if len(sub):
        r = sub.iloc[0]
        print(f"{f:16s} co2={r['co2_estimate']:6.1f} g/km  "
              f"af={r['attribution_factor']:.2f}  -> {r['financed_tco2']:.3f} tCO2e/yr")

petrol           co2= 162.3 g/km  af=0.84  -> 1.232 tCO2e/yr
diesel           co2= 181.0 g/km  af=0.10  -> 0.253 tCO2e/yr
electric         co2=   0.0 g/km  af=0.80  -> 0.000 tCO2e/yr
petrol/electric  co2=  28.8 g/km  af=0.56  -> 0.210 tCO2e/yr


## 7. Sensitivity: annual distance

Mileage is the second unknown on the vehicle side, and unlike gCO2/km it is *not*
modelled — a loan record carries no signal about how far this driver will go, only
population averages exist. That makes it the biggest single assumption in the
total, so we report a band, not a point. Here we scale every mileage assumption
+/-20%.

In [7]:
base_table = pf.DEFAULT_MILEAGE
for factor in [0.8, 0.9, 1.0, 1.1, 1.2]:
    scaled = {k: v * factor for k, v in base_table.items()}
    t = mod.copy()
    t["financed_tco2"] = pf.financed_emissions_physical(t, mileage_table=scaled)
    tot = t["financed_tco2"].sum()
    print(f"  mileage x{factor:.1f}  ->  {tot:,.0f} tCO2e/yr")

  mileage x0.8  ->  2,864 tCO2e/yr
  mileage x0.9  ->  3,221 tCO2e/yr
  mileage x1.0  ->  3,579 tCO2e/yr
  mileage x1.1  ->  3,937 tCO2e/yr
  mileage x1.2  ->  4,295 tCO2e/yr


## 8. Optional: electric well-to-wheel

Tailpipe (tank-to-wheel) EV emissions are 0 by construction, which is the
PCAF/peer-comparable convention. Toggling well-to-wheel charges grid emissions
instead — small on France's low-carbon grid, but the switch shows tailpipe is a
convention, not a life-cycle truth.

In [8]:
ttw = pf.financed_emissions_physical(mod).sum()
wtw = pf.financed_emissions_physical(mod, ev_wtw=True).sum()
print(f"tank-to-wheel : {ttw:,.0f} tCO2e/yr")
print(f"well-to-wheel : {wtw:,.0f} tCO2e/yr  ({100*(wtw/ttw-1):+.1f}%)")

tank-to-wheel : 3,579 tCO2e/yr
well-to-wheel : 3,645 tCO2e/yr  (+1.8%)


## Summary

- **Attribution** (outstanding / origination value) decides the lender's share of
  each car's emissions; capped at 1.
- **Physical method** (Model A gCO2/km × assumed km) is the score-3 estimate;
  the **economic method** (loan amount only) is the score-5 fallback it replaces.
- On the same book the two methods disagree by a wide margin — the reason PCAF
  grades data quality at all.
- **Mileage** is the dominant assumption and is reported as a band, not a point.

The finding that carries the project: Model A is not a redundant regression of a
column that already exists. Joined to a loan book it is a **PCAF data-quality
upgrade** — the exact tool a lender like BNPPF needs to move an auto portfolio off
the economic fallback and onto a defensible, physics-grounded number.